In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Dense, GRU, Input, Dropout, Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam


N = 10000
data = np.array([np.sin(x/20) for x in range(N)]) + 0.1*np.random.randn(N)
plt.plot(data[:100])


off = 3
length = off*2+1
X = np.array([ np.diag(np.hstack((data[i:i+off], data[i+off+1:i+length]))) for i in range(N-length)])
Y = data[off:N-off-1]
print(X.shape, Y.shape, sep='\n')


model = Sequential()
model.add(Input((length-1, length-1)))
model.add( Bidirectional(GRU(2)) )
model.add(Dense(1, activation='linear'))
model.summary()

model.compile(loss='mean_squared_error', optimizer=Adam(0.01))


history = model.fit(X, Y, batch_size=32, epochs=10)

M = 200
XX = np.zeros(M)
XX[:off] = data[:off]
for i in range(M-off-1):
    x = np.diag( np.hstack( (XX[i:i+off], data[i+off+1:i+length])) )
    x = np.expand_dims(x, axis=0)
    y = model.predict(x)
    XX[i+off] = y

plt.plot(XX[:M])
plt.plot(data[:M])

## Izah

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# TensorFlow-un artıq xəbərdarlıq mesajlarını gizlədir.
# DİQQƏT: TensorFlow import olunmazdan ƏVVƏL yazılmalıdır.


import numpy as np
# NumPy — massivlərlə işləmək üçün.
# Bu kodda:
# - np.sin       → sinus dalğası yaratmaq
# - np.random.randn → səs-küy (noise) əlavə etmək
# - np.diag      → vektordan diaqonal matris düzəltmək
# - np.hstack    → iki massivi yan-yana birləşdirmək
# - np.expand_dims → massivə əlavə ölçü (batch ölçüsü) əlavə etmək


import matplotlib.pyplot as plt
# matplotlib — qrafik çəkmək üçün.
# plt.plot(...) → xətt qrafiki çəkir.
# DİQQƏT: kodda plt.show() yoxdur. Jupyter / Colab-da qrafik özü görünür,
# amma adi .py faylı kimi işlədəndə qrafiki görmək üçün sonda
# plt.show() yazmaq lazımdır.


from tensorflow.keras.layers import Dense, GRU, Input, Dropout, Bidirectional
# Keras qatları:
#
# Dense         → tam bağlı qat (çıxış qatı kimi).
# GRU           → rekurrent qat (Gated Recurrent Unit): 2 qapı (update, reset),
#                 yalnız 1 gizli vəziyyət (h). LSTM-dən yüngül və sürətlidir.
# Input         → modelin giriş formasını elan edir. Bu kodda İSTİFADƏ OLUNUR
#                 (əvvəlki kodda istifadə olunmurdu).
# Dropout       → import olunub, amma İSTİFADƏ OLUNMUR
#                 (overfitting-ə qarşı əlavə etmək olardı).
# Bidirectional → İKİ İSTİQAMƏTLİ sarğı (wrapper). Daxilindəki qatı (burada
#                 GRU-nu) iki dəfə işlədir: bir dəfə ardıcıllığı SOLDAN SAĞA,
#                 bir dəfə SAĞDAN SOLA oxuyur, sonra nəticələri birləşdirir.
#                 Bu kodda çox yerinə düşür: çünki biz ortadakı nöqtəni
#                 həm keçmişə, həm gələcəyə baxaraq təxmin edirik.


from tensorflow.keras.models import Sequential
# Sequential model: qatlar ardıcıl düzülür.


from tensorflow.keras.optimizers import Adam
# Adam optimizatoru. Obyekt kimi import edirik ki, öyrənmə sürətini
# (learning rate) özümüz təyin edə bilək.


# =====================================================================
# BU KOD NƏDİR? — ZAMAN SERİYASI: BOŞLUQ DOLDURMA (Bidirectional GRU ilə)
# =====================================================================
#
# Məqsəd: səs-küylü (noise-lu) sinus dalğası var. Hər nöqtə üçün onun
# SOLUNDAKI 3 və SAĞINDAKI 3 qonşuya baxıb ORTADAKI nöqtənin qiymətini
# təxmin etmək. Yəni model "boşluğu doldurur".
#
# Bu, REGRESSİYA məsələsidir: model sinif yox, HƏQİQİ ƏDƏD təxmin edir.
#
# SENTIMENT KODU İLƏ MÜQAYİSƏ:
#
#                    Sentiment kodu              Bu kod
# Məsələ:            təsnifat (2 sinif)          regressiya (1 ədəd)
# Giriş:             söz nömrələri               həqiqi ədədlər
# Embedding qatı:    VAR                         YOXDUR (ədədlər artıq ədəddir)
# Rekurrent qat:     GRU(128) + GRU(64)          Bidirectional(GRU(2))
# Çıxış qatı:        Dense(2, softmax)           Dense(1, linear)
# Loss:              categorical_crossentropy    mean_squared_error
# Metrik:            accuracy                    yoxdur (loss özü kifayətdir)
# Öyrənmə sürəti:    0.0001                      0.01
# Epoch:             50                          10
#
# Niyə accuracy yoxdur? Çünki ədəd təxmin edirik: "düzgün/yanlış" demək
# mənasızdır (0.5012 təxmin edib, cavab 0.5000 idi — bu düzgündür, yoxsa yox?).
# Əvəzində səhvin ortalama kvadratı (MSE) ölçülür.
# Niyə softmax yox, linear? Çıxış istənilən ədəd ola bilər (müsbət, mənfi),
# softmax isə yalnız 0–1 arası ehtimal verərdi.


# ---------------------------------------------------------------------
# 1) DATANIN YARADILMASI
# ---------------------------------------------------------------------

N = 10000
# Cəmi neçə zaman nöqtəsi yaradacağımız.

data = np.array([np.sin(x/20) for x in range(N)]) + 0.1*np.random.randn(N)
# Səs-küylü sinus dalğası.
#
# np.sin(x/20) → x = 0, 1, 2, ..., 9999 üçün sinus. Qiymətlər -1 ilə +1
#   arasındadır. "/20" dalğanı yavaşladır: bir tam dövr təxminən
#   2·π·20 ≈ 125.7 nöqtə çəkir.
#
# 0.1*np.random.randn(N) → standart normal paylanmadan N təsadüfi ədəd,
#   0.1-ə vurulub. Yəni səs-küyün standart kənarlaşması 0.1-dir
#   (dispersiyası 0.01). Bu səs-küy hər nöqtəyə əlavə olunur.
#
# NƏTİCƏ: data = "təmiz sinus + kiçik təsadüfi titrəmə".
# Forma: (10000,)
#
# DİQQƏT: np.random.seed(...) yoxdur, ona görə hər işlətdikdə səs-küy
# fərqli olur və nəticələr bir az dəyişir.
# (Qısa yazılışı: np.sin(np.arange(N)/20) + 0.1*np.random.randn(N))

plt.plot(data[:100])
# İlk 100 nöqtəni çəkir. Burada sinusun təxminən 0.8 dövrünü görürük:
# hamar dalğa, amma üstündə kiçik təsadüfi titrəmələr.
# Məqsəd: datanın düzgün yarandığını gözlə yoxlamaq.


# ---------------------------------------------------------------------
# 2) NÜMUNƏLƏRİN (X) VƏ CAVABLARIN (Y) HAZIRLANMASI
# ---------------------------------------------------------------------

off = 3
# "offset" — hədəf nöqtədən hər iki tərəfə neçə qonşuya baxırıq.
# 3 sol + 3 sağ = 6 qonşu.

length = off*2+1
# Pəncərənin tam uzunluğu: 3 + 1 (hədəf) + 3 = 7.
#
# Pəncərə belə görünür (i — pəncərənin başlanğıc indeksidir):
#
#   indeks:   i   i+1  i+2 | i+3 | i+4  i+5  i+6
#             └─ keçmiş ─┘   HƏDƏF   └─ gələcək ─┘
#             (3 nöqtə)    (təxmin    (3 nöqtə)
#                          edilən)
#
# Model ortadakı i+3-ün qiymətini görmür, onu yalnız 6 qonşudan tapmalıdır.

X = np.array([ np.diag(np.hstack((data[i:i+off], data[i+off+1:i+length]))) for i in range(N-length)])
# Giriş datası. Bu sətir çox sıxdır, addım-addım açaq:
#
# a) data[i:i+off]            → sol 3 qonşu  (indekslər i, i+1, i+2)
# b) data[i+off+1:i+length]   → sağ 3 qonşu  (indekslər i+4, i+5, i+6)
#    Ortadakı data[i+3] ATLANIR — çünki o bizim hədəfdir.
# c) np.hstack((a, b))        → ikisini birləşdirir: 6 ədədlik vektor.
#    Məsələn: [-0.12, 0.05, 0.11, 0.31, 0.38, 0.47]
# d) np.diag(vektor)          → 6 ədədlik vektordan 6x6 DİAQONAL MATRİS düzəldir:
#
#        [-0.12   0     0     0     0     0  ]
#        [  0    0.05   0     0     0     0  ]
#        [  0     0    0.11   0     0     0  ]
#        [  0     0     0    0.31   0     0  ]
#        [  0     0     0     0    0.38   0  ]
#        [  0     0     0     0     0    0.47]
#
# NİYƏ BELƏ EDİLİR? GRU ardıcıllıq gözləyir: forma (zaman_addımı, xüsusiyyət).
# Burada 6 zaman addımı var və hər addımda GRU bir sətir görür:
#   1-ci addım: [-0.12, 0, 0, 0, 0, 0]   → yalnız 1-ci qonşu
#   2-ci addım: [0, 0.05, 0, 0, 0, 0]    → yalnız 2-ci qonşu
#   ... və s.
# Yəni hər addımda "dəyər + onun mövqeyi (hansı sütunda olduğu)" verilir.
# Bu, bir növ "dəyər × one-hot(mövqe)" kodlamasıdır.
#
# DİQQƏT: bu məcburi deyil. Daha sadə üsul: hər nümunəni (6, 1) formasında
# vermək (6 addım, hər addımda 1 ədəd). Onda parametr sayı da azalardı.
# Diaqonal üsul modelə mövqe məlumatını açıq şəkildə verir.
#
# range(N-length) → i = 0, 1, ..., 9992 (cəmi 9993 pəncərə).
# Niyə N-length? Pəncərə sondan kənara çıxmasın deyə.
#
# Forma: (9993, 6, 6) → (nümunə_sayı, zaman_addımı, xüsusiyyət)

Y = data[off:N-off-1]
# Cavablar (hədəflər): hər pəncərənin ORTASINDAKI nöqtə.
#
# i-ci pəncərənin ortası data[i+3]-dür. Deməli Y[i] = data[i+3].
# Y[0] = data[3], Y[1] = data[4], ..., ona görə dilim data[3 : N-4].
# Uzunluq: (N-4) - 3 = N-7 = 9993 — X ilə EYNİDİR. (Bu, vacibdir:
# X və Y-də nümunə sayı üst-üstə düşməlidir.)
#
# DİQQƏT: Y-dəki qiymətlər də səs-küylüdür. Yəni model "təmiz sinusu"
# yox, "səs-küylü nöqtəni" təxmin etməyə öyrədilir. Səs-küyün özünü
# heç vaxt dəqiq təxmin etmək olmaz (təsadüfidir), ona görə loss
# sıfıra düşməyəcək. Aşağıda buna baxın.

print(X.shape, Y.shape, sep='\n')
# Formaları yoxlayır:
#   (9993, 6, 6)
#   (9993,)
# Birinci rəqəm (nümunə sayı) hər ikisində eyni olmalıdır.
# sep='\n' → iki çıxışı ayrı sətirlərdə göstərir.


# ---------------------------------------------------------------------
# 3) MODELİN QURULMASI
# ---------------------------------------------------------------------

model = Sequential()

model.add(Input((length-1, length-1)))
# Giriş qatı: forma (6, 6) → (zaman_addımı=6, xüsusiyyət=6).
# length-1 = 6. Batch ölçüsü (nümunə sayı) göstərilmir.
# Bu qat parametr saxlamır, yalnız modelə hansı formada data gələcəyini deyir.
# (Embedding-li əvvəlki kodda input_length bunun rolunu oynayırdı.)

model.add( Bidirectional(GRU(2)) )
# İKİ İSTİQAMƏTLİ GRU qatı.
#
# Daxildə GRU(2): cəmi 2 neyron — çox kiçikdir, çünki məsələ sadədir
# (hamar sinus dalğası).
#
# Bidirectional nə edir?
#   - İRƏLİ GRU: ardıcıllığı 1-ci addımdan 6-cıya oxuyur
#                (soldakı qonşulardan sağdakılara).
#   - GERİ GRU:  ardıcıllığı 6-cı addımdan 1-ciyə oxuyur
#                (sağdakı qonşulardan soldakılara).
#   Hər biri son vəziyyətini (2 ədəd) verir, sonra bu iki vektor
#   BİRLƏŞDİRİLİR (concatenate): 2 + 2 = 4 ədəd.
#   Bu 4 ədəd bütün pəncərənin xülasəsidir.
#
# return_sequences yoxdur (False) → yalnız SON vəziyyət qaytarılır.
#
# Forma: (6, 6) → (4,)
#
# Parametr sayı (bir GRU üçün, Keras-ın defolt reset_after=True rejimi):
#   3 × (giriş×vahid + vahid×vahid + 2×vahid)
#   = 3 × (6×2 + 2×2 + 2×2) = 3 × 20 = 60
# İki istiqamət olduğu üçün: 60 × 2 = 120.

model.add(Dense(1, activation='linear'))
# Çıxış qatı: 1 neyron — təxmin edilən ədədin özü.
# activation='linear' → heç bir çevrilmə yoxdur, çıxış istənilən ədəd ola bilər.
# (Linear defolt olduğu üçün yazmasaq da eyni olardı; açıq yazmaq oxunaqlıdır.)
#
# Forma: (4,) → (1,)
# Parametr: 4 × 1 + 1 = 5

model.summary()
# Modelin quruluşunu çap edir.
# Ümumi parametr: 120 + 5 = 125. Çox kiçik modeldir.
# (Sentiment kodunda 264450 idi.) Rəqəmlər köhnə Keras versiyasında
# (reset_after=False) bir az fərqli çıxa bilər.

model.compile(loss='mean_squared_error', optimizer=Adam(0.01))
# loss='mean_squared_error' (MSE) → orta kvadratik səhv:
#   MSE = ortalama( (təxmin - həqiqi)² )
# Regressiya üçün standart seçimdir. Böyük səhvləri kvadrata yüksəltdiyi
# üçün daha çox cəzalandırır.
#
# optimizer=Adam(0.01) → öyrənmə sürəti 0.01. Defolt (0.001)-dən 10 dəfə
# BÖYÜKDÜR: model cəmi 125 parametrdən ibarətdir və 10 epoch-da
# öyrənməlidir, ona görə sürətli addımlar atırıq.
#
# DİQQƏT: metrics yoxdur, çünki regressiyada accuracy mənasızdır.
# Öyrənməni loss-dan izləyirik.


# ---------------------------------------------------------------------
# 4) ÖYRƏTMƏ
# ---------------------------------------------------------------------

history = model.fit(X, Y, batch_size=32, epochs=10)
# Öyrətmə: hər addımda 32 nümunə, bütün data 10 dəfə keçilir.
# Hər epoch-da 9993 / 32 ≈ 313 addım var → cəmi təxminən 3130 çəki yeniləməsi.
#
# NÜMUNƏLƏRİN QARIŞDIRILMASI: sentiment kodunda əl ilə qarışdırırdıq.
# Burada ehtiyac yoxdur, çünki model.fit defolt olaraq shuffle=True ilə
# hər epoch-da nümunələri özü qarışdırır. (Əvvəlki koddakı qarışdırma
# addımı yenə də zərərsizdir, sadəcə məcburi deyil.)
#
# NƏ GÖZLƏMƏK LAZIMDIR? loss əvvəlcə böyük olur, sonra sürətlə düşür və
# təxminən 0.01 civarında dayanır. Niyə sıfıra düşmür?
#   Hədəf nöqtənin üzərindəki səs-küyün dispersiyası 0.1² = 0.01-dir.
#   Bu səs-küy TƏSADÜFİDİR, qonşulara baxmaqla bilmək mümkün deyil.
#   Yəni 0.01 təxminən "mümkün olan ən yaxşı nəticənin" yaxınlığıdır.
#   (Qonşulardakı səs-küy səbəbindən real loss bir az yuxarı olur,
#   məsələn 0.011–0.015.) Loss bundan çox aşağı düşərsə, model səs-küyü
#   əzbərləyir (overfitting) deməkdir.
#
# DİQQƏT: validation_split yoxdur, yəni yoxlama datası ayrılmayıb.
# Bu məsələdə overfitting riski aşağıdır (125 parametr, 9993 nümunə),
# amma düzgün yoxlama üçün: model.fit(..., validation_split=0.2).
# history → hər epoch-un loss tarixçəsi; plt.plot(history.history['loss'])
# ilə öyrənmə əyrisini çəkmək olar.


# ---------------------------------------------------------------------
# 5) PROQNOZ: SİNUS DALĞASININ BƏRPASI
# ---------------------------------------------------------------------

M = 200
# Bərpa edəcəyimiz nöqtələrin sayı: ilk 200 nöqtə (təxminən 1.6 dövr).

XX = np.zeros(M)
# Nəticəni yığacağımız boş massiv (hamısı 0). Forma: (200,)

XX[:off] = data[:off]
# İlk 3 nöqtəni HƏQİQİ datadan köçürür.
# Niyə? İlk pəncərə üçün 3 sol qonşu lazımdır, hələ heç nə təxmin
# etməmişik, ona görə başlanğıc kimi real qiymətləri veririk.

for i in range(M-off-1):
    # i = 0, 1, ..., 195 (cəmi 196 addım). M-off-1 = 196.
    # Hər addımda bir nöqtəni təxmin edib XX-ə yazırıq.

    x = np.diag( np.hstack( (XX[i:i+off], data[i+off+1:i+length])) )
    # Eyni pəncərə quruluşu, amma İKİ FƏRQLƏ:
    #   - SOL 3 qonşu XX-dən götürülür (yəni modelin ÖZ təxminlərindən;
    #     yalnız ilk addımda real dəyərlərdir, çünki XX[:3] real idi).
    #   - SAĞ 3 qonşu data-dan götürülür (HƏQİQİ, səs-küylü qiymətlər).
    # Forma: (6, 6)
    #
    # Yəni keçmiş "özümüzün qurduğumuz" nəticədir, gələcək isə real datadır.
    # Bu, zəncirvari proses kimidir: hər təxmin növbəti addımda sol
    # qonşu kimi istifadə olunur. Kiçik səhvlər toplana bilər.

    x = np.expand_dims(x, axis=0)
    # Əvvəlinə batch ölçüsü əlavə edir: (6, 6) → (1, 6, 6).
    # Keras həmişə "nümunələr toplusu" gözləyir, tək nümunə olsa belə.

    y = model.predict(x)
    # Modelin təxmini. Forma: (1, 1), məsələn [[0.2314]].
    # DİQQƏT: 196 dəfə ayrıca predict çağırmaq YAVAŞDIR (hər çağırışın
    # öz yükü var). Sürətləndirmək üçün model.predict(x, verbose=0)
    # yazıb çıxış mətnini söndürmək olar.

    XX[i+off] = y
    # Təxmini ortadakı mövqeyə yazır (indekslər 3, 4, 5, ..., 198).
    #
    # DİQQƏT 1: y forma (1, 1) olan massivdir, tək ədədə mənimsədilir.
    # Yeni NumPy versiyalarında bu DeprecationWarning verir.
    # Təmiz yazılış: XX[i+off] = y[0, 0].
    #
    # DİQQƏT 2: dövr 196 dəfə işləyir, deməli yazılan indekslər 3-dən
    # 198-ə qədərdir. XX[199] heç vaxt doldurulmur və 0 olaraq QALIR.
    # Qrafikdə sonda kiçik "düşmə" görsəniz səbəbi budur, bu model xətası deyil.

plt.plot(XX[:M])
# Modelin bərpa etdiyi əyrini çəkir.

plt.plot(data[:M])
# Müqayisə üçün orijinal (səs-küylü) datanı eyni qrafikdə çəkir.
#
# NƏ GÖRMƏLİYİK? Model əyrisi (XX) səs-küylü əyrinin (data) ORTASINDAN
# keçən HAMAR sinus dalğasına yaxın olmalıdır. Çünki model səs-küyü
# təxmin edə bilmir, yalnız ümumi tendensiyanı (sinusu) öyrənir.
# Bu, əslində bir növ "səs-küyün təmizlənməsi" (denoising) effektidir.
#
# DİQQƏT: yoxlama öyrədilmiş datanın ÜZƏRİNDƏ aparılır (ilk 200 nöqtə
# öyrətmə datasının içindədir). Daha dürüst test üçün datanı ikiyə bölüb
# modeli öyrətmədiyi hissədə sınamaq lazımdır, məsələn data[8000:8200].
# Sinus kimi sadə funksiyada nəticə yenə yaxşı olacaq, amma real
# tapşırıqlarda bu fərq vacibdir.